# Credit union benchmark at scale

Same generate → verify → score loop as notebooks 2–4 and the SaaS notebook, on the `mock_credit_union` card servicing domain.

- Generate a balanced set of tasks (task types × personalities), then keep only verified ones.
- Evaluate the multi-agent pipeline, compare two agent models, and look at results by persona.

**What is different from the SaaS notebook**
- 6 task types × 5 personalities = 30 drafts.
- Several task types have only 3–4 eligible customers (for example, refusals need a transaction older than 60 days). When a customer can't be reused, the notebook reuses one and prints a note.

**Lessons to check for this domain**
- Persona changes the conversation. A customer who raises fraud can push the agent onto the fraud path, which changes the expected database writes.
- Outcome scoring (final DB + required phrases) only tells you pass or fail. Read the traces to see whether the agent or the expected outcome is wrong.
- `pass@1` varies from run to run. Run the same tasks several times before deciding a task is broken.

In [ ]:
import json
from collections import Counter, defaultdict

from aieng.syn_data.synbench.agents.pipeline import DEFAULT_ROLES, AgentPipeline
from aieng.syn_data.synbench.display import (  # noqa: E402
    show_actions,
    show_benchmark_by_task_type,
    show_pipeline_messages,
    show_session,
    show_task,
)
from aieng.syn_data.synbench.evaluation.metrics import MetricsCollector
from aieng.syn_data.synbench.llm import get_client
from aieng.syn_data.synbench.llm.config import get_agent_model, get_user_simulator_model
from utils.utils import get_root_directory


ROOT = get_root_directory()
DOMAIN_PATH = ROOT / "domains" / "mock_credit_union"
OUT_DIR = ROOT / "data" / "benchmarks" / "mock_credit_union"
OUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR = OUT_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Implementation root:", ROOT)
print("Domain:", DOMAIN_PATH)
print("Output:", OUT_DIR)

## Step 1 — Load and inspect the domain

Validate the complete domain bundle, then report aggregate sizes instead of printing the full database.

In [ ]:
from aieng.syn_data.synbench.domain.loader import load_domain, validate_domain


errors = validate_domain(DOMAIN_PATH)
assert errors == [], f"Domain validation failed: {errors}"
domain = load_domain(DOMAIN_PATH)

print("Domain:", domain.manifest.name)
print("\nTask types:", list(domain.task_types.keys()))
print("Seed tasks:", len(domain.seed_tasks))
for collection in ("customers", "accounts", "cards", "transactions", "fees", "case_index"):
    print(f"{collection}: {len(domain.db.get(collection, {}))}")

sample_customer = next(iter(domain.db["case_index"].values()))
print("\nSample case_index entry:\n", json.dumps(sample_customer, indent=2))

## Step 2 — Inspect and replay seed tasks

Each task type has hand-verified seeds. Replay one oracle to show the target database state used for scoring.

In [ ]:
from aieng.syn_data.synbench.environment.core import replay_actions
from aieng.syn_data.synbench.environment.hashing import db_hash


for task_type in domain.task_types:
    ids = [task.id for task in domain.seed_tasks if task.task_type == task_type]
    print(f"{task_type}: {ids}")

seed = domain.seed_tasks[0]
target_env = replay_actions(domain, seed.evaluation_criteria.actions)
print("\nReplayed seed:", seed.id)
print("Target DB hash:", db_hash(target_env.db))

## Step 3 — Generate tasks with balanced constraints

Hold the task type fixed while sampling an eligible customer for each personality. The sampler already applies the eligibility rules in `generation.yaml`. This notebook also tries to avoid reusing a customer within the same task type, and falls back to reuse when no other eligible customer is found.

With the defaults this requests 30 drafts: 6 task types × 5 personalities.

**Don't run this cell if you have already generated the tasks.**

In [ ]:
from aieng.syn_data.synbench.generation.generator import TrajectoryGenerator
from aieng.syn_data.synbench.generation.sampler import ConstraintSampler


TASK_TYPES = list(domain.task_types.keys())
PERSONALITIES = [style["name"] for style in domain.user_simulator["personality_styles"]]
MAX_SAMPLE_ATTEMPTS = 100

sampler = ConstraintSampler(domain, seed=42)
generator = TrajectoryGenerator(domain, seed=42)
drafts = []
generation_rows = []
for task_type in TASK_TYPES:
    used_customers = set()
    for personality in PERSONALITIES:
        constraints = None
        for _ in range(MAX_SAMPLE_ATTEMPTS):
            candidate = sampler.sample(task_type=task_type, personality_name=personality)
            if candidate.primary_id not in used_customers:
                constraints = candidate
                break
        if constraints is None:
            constraints = candidate
            print(f"Note: reusing a customer for {task_type}/{personality} (no other eligible customer found)")
        used_customers.add(constraints.primary_id)

        task_id = f"scale_{task_type}_{personality}"
        draft = generator.generate_one(task_id=task_id, constraints=constraints)
        drafts.append(draft)
        generation_rows.append(
            {
                "task_type": task_type,
                "customer_id": constraints.primary_id,
                "user_name": draft.user_scenario.user_name,
                "personality": personality,
            },
        )

print(f"Generated {len(drafts)} drafts")
for row in generation_rows:
    print(f"{row['task_type']:<24} {row['personality']:<16} {row['customer_id']:<10} {row['user_name']}")

## Step 4 — Verify and save

Apply domain rules, write rules, and replay. Only passing tasks are saved.

In [ ]:
from aieng.syn_data.synbench.generation.generator import GenerationRun


run = GenerationRun(domain, seed=42)
verified_tasks, rejections = run.verify_drafts(drafts)
task_path = run.write_tasks(verified_tasks, OUT_DIR)

print("Drafts by type:", dict(Counter(task.task_type for task in drafts)))
print("Passing by type:", dict(Counter(task.task_type for task in verified_tasks)))
print("Rejected by type:", dict(Counter(task.task_type for task, _ in rejections)))
print("Saved", len(verified_tasks), "tasks to", task_path)

for task, reasons in rejections:
    print(f"REJECTED {task.id}: {'; '.join(reasons)}")

## Step 5 — Load and re-verify saved tasks

Reload the saved JSON through the task schema and verify every task again before evaluation.

**Don't forget to load and inspect the domain before running this cell.**

In [ ]:
from aieng.syn_data.synbench.schemas.tasks import Task
from aieng.syn_data.synbench.verification.pipeline import verify_draft


TASK_PATH = OUT_DIR / "tasks.json"

with TASK_PATH.open(encoding="utf-8") as f:
    saved = json.load(f)

filtered_tasks = []
for item in saved["tasks"]:
    task = Task.model_validate(item)
    verification = verify_draft(domain, task)
    status = "OK" if verification.verification_report.passed else "FAIL"
    print(f"[{status}] {task.id}")
    if verification.verification_report.passed:
        filtered_tasks.append(task)

print(f"Ready to evaluate: {len(filtered_tasks)} tasks")

## Step 6 — Run one task and inspect it

Before running the full benchmark, run one task through the multi-agent pipeline (user simulator → planner → executor → critic) and read the whole conversation. Check whether the agent's behaviour is wrong or whether the expected actions in the oracle are missing something the policy requires.

In [ ]:
agent_model = get_agent_model()
agent_client = get_client(agent_model)
user_model = get_user_simulator_model()
user_client = get_client(user_model)
print("Agent Model:", agent_model)
print("User Model:", user_model)
print("Role order:", DEFAULT_ROLES)

pipeline = AgentPipeline(
    domain,
    max_turns=12,
    max_dialogue_turns=5,
    agent_client=agent_client,
    user_simulator_client=user_client,
)

sample_task = filtered_tasks[0]
show_task(sample_task, title=f"Sample task: {sample_task.id}")

sample_session = pipeline.run_task(sample_task)
sample_score = pipeline.score_session(sample_session)
print("Task:", sample_task.id)
print("Reward:", sample_score.reward)
print("DB reward:", sample_score.db_reward)
print("Communication reward:", sample_score.communicate_reward)
if sample_score.missing_communicate:
    print("Missing communicate:", sample_score.missing_communicate)

show_session(sample_session, title="Session (after dialogue)", show_chat=False)
show_pipeline_messages(sample_session, title="Pipeline messages")
show_actions(sample_session.agent_actions, title="Agent actions")

## Step 7 — Run the full benchmark

This can spend API credits. Each task is run once, so `pass@1` can change between runs. For a more reliable number, run the benchmark several times and compare.

In [ ]:
def run_full_benchmark(agent_model_name):
    result_dir = RESULTS_DIR / agent_model_name
    result_dir.mkdir(parents=True, exist_ok=True)
    agent_pipeline = AgentPipeline(
        domain,
        max_turns=12,
        max_dialogue_turns=5,
        agent_client=get_client(agent_model_name),
        user_simulator_client=get_client(get_user_simulator_model()),
    )
    metrics = MetricsCollector()
    scores = {}
    for task in filtered_tasks:
        score = agent_pipeline.run_and_score_task(task)
        scores[task.id] = score
        metrics.add(task.id, score)
        persona = task.user_scenario.personality_style or "unknown"
        print(f"{task.id:<40} persona={persona:<15} reward={score.reward:.2f} db={score.db_reward:.2f} comm={score.communicate_reward:.2f}")
    summary = metrics.summary()
    with open(result_dir / "benchmark_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)
    print("\nSaved benchmark summary at", result_dir / "benchmark_summary.json")
    return metrics, scores, agent_pipeline


metrics_default, scores_default, pipeline_default = run_full_benchmark(agent_model)

In [ ]:
show_benchmark_by_task_type(
    metrics_default,
    filtered_tasks,
    agent_model=agent_model,
    task_types=TASK_TYPES,
)

with open(RESULTS_DIR / agent_model / "benchmark_summary.json", "r", encoding="utf-8") as f:
    summary = json.load(f)
print("n_tasks:", summary["n_tasks"])
print("pass_at_1:", summary["pass_at_1"])
print("mean_db_reward:", summary["mean_db_reward"])
print("mean_communicate_reward:", summary["mean_communicate_reward"])
print("mean_dialogue_turns:", summary["mean_dialogue_turns"])

## Step 8 — Does persona change the outcome?

Group the pass rate by the simulated customer's personality. A large gap for one task type suggests the expected outcome doesn't cover how that persona behaves.

In [ ]:
by_persona = defaultdict(list)
for task in filtered_tasks:
    persona = task.user_scenario.personality_style or "unknown"
    by_persona[persona].append(scores_default[task.id].reward)

for persona, rewards in sorted(by_persona.items()):
    passed = sum(1 for r in rewards if r >= 1.0)
    print(f"{persona:<16} pass={passed}/{len(rewards)}")

## Step 9 — Inspect a failed task

Pick the first failed task, compare the agent's actions with the oracle's expected actions, and read the conversation. Decide whether the agent made a mistake or the expected outcome is too narrow.

In [ ]:
failed_ids = [task_id for task_id, s in scores_default.items() if s.reward < 1.0]
print("Failed tasks:", failed_ids)

if failed_ids:
    failed_task = next(task for task in filtered_tasks if task.id == failed_ids[0])
    show_task(failed_task, title=f"Failed task: {failed_task.id}")

    print("\nExpected (oracle) actions:")
    for action in failed_task.evaluation_criteria.actions:
        print("  ", action.name, action.arguments)

    failed_session = pipeline_default.run_task(failed_task)
    failed_score = pipeline_default.score_session(failed_session)
    print("\nRe-run reward:", failed_score.reward, "db:", failed_score.db_reward, "comm:", failed_score.communicate_reward)
    show_actions(failed_session.agent_actions, title="Agent actions (re-run)")
    show_pipeline_messages(failed_session, title="Pipeline messages (re-run)")

## Step 10 — Run the benchmark with a second agent

Compare a second model on the same tasks. Change `second_agent_model` to any model configured in `.env`.

In [ ]:
second_agent_model = "gpt-4o"
metrics_second, scores_second, _ = run_full_benchmark(second_agent_model)

show_benchmark_by_task_type(
    metrics_second,
    filtered_tasks,
    agent_model=second_agent_model,
    task_types=TASK_TYPES,
)